# TP dynamique côtière: exercice de décision à cinq facteurs PAK/Kribi

**GGOSSS 2026 · Jour 2 · Instructeur: Nourdi Njutapvoui**

Question de décision: **si vous êtes responsable du suivi côtier à PAK/Kribi, quel secteur faut-il suivre en priorité, et pourquoi?**

Le TP combine cinq facteurs: **Shoreline**, **Surface Water Velocity (SWV)**, **Tide**, **Wave** et **Wind**. Le résultat attendu est une recommandation de suivi, pas seulement un graphique.

## Objectifs pédagogiques

À la fin de ce TP, les participants seront capables de:

- combiner des indicateurs de shoreline, courant de surface, marée, vague et vent;
- identifier les secteurs en érosion, accrétion et stabilité;
- construire et critiquer un indice de pression transparent à cinq facteurs;
- tester comment les choix de pondération modifient les priorités de suivi;
- communiquer l'incertitude avant de formuler une recommandation de gestion.

In [ ]:
# Bloc de préparation GGOSSS 2026: importer les packages, définir la palette graphique et localiser les données.
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

GGOSSS_COLORS = {
    "navy": "#17304f",
    "shoreline": "#c62828",
    "swv": "#0077b6",
    "tide": "#6a1b9a",
    "wave": "#00a6a6",
    "wind": "#ef6c00",
    "neutral": "#607d8b",
}

plt.rcParams.update({
    "figure.figsize": (8, 4),
    "axes.grid": True,
    "axes.facecolor": "#fbfcfd",
    "axes.edgecolor": GGOSSS_COLORS["navy"],
    "axes.titleweight": "bold",
    "axes.titlesize": 12,
})

def find_data_dir():
    """Find the data folder in either the GitHub layout or the local instructor layout."""
    for candidate in [Path("../data"), Path("../Datasets"), Path("data"), Path("Datasets")]:
        if candidate.exists():
            return candidate
    raise FileNotFoundError("Could not find data directory. Run from the notebook folder or the session folder.")

DATA_DIR = find_data_dir()
DATA_DIR

In [ ]:
# Bloc de chargement des données: lire les tables légères PAK/Kribi dérivées du jeu complet instructeur.
rates = pd.read_csv(DATA_DIR / "pak_shoreline_change_rates_2026.csv")
boxes = pd.read_csv(DATA_DIR / "pak_coastsat_box_inventory.csv")
transects = pd.read_csv(DATA_DIR / "pak_coastsat_transect_inventory.csv")
factors = pd.read_csv(DATA_DIR / "pak_integrated_coastal_factors.csv")

print("DSAS shoreline rates:", rates.shape)
print("CoastSat boxes:", boxes.shape)
print("CoastSat transects:", transects.shape)
print("Integrated five-factor table:", factors.shape)
factors.head()

In [ ]:
# Bloc cartographique: situer PAK/Kribi avec un vrai fond continental et un zoom local.
# L'objectif est de relier les facteurs Shoreline, SWV, marée, vagues et vent à une géographie lisible.
from matplotlib.patches import Rectangle

try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    CARTOPY_DISPONIBLE = True
except Exception:
    CARTOPY_DISPONIBLE = False

localities_path = DATA_DIR / "selected_localities_for_maps.csv"
localities = pd.read_csv(localities_path) if localities_path.exists() else pd.DataFrame(columns=["Name", "Lon", "Lat", "Zone"])

extent_local = [9.55, 10.15, 2.15, 3.35]
extent_regional = [7.2, 12.7, 0.8, 5.2]

def ajouter_fond_geographique(ax, extent, titre):
    """Ajouter continent, océan, trait de côte, frontières et grille géographique."""
    if CARTOPY_DISPONIBLE:
        ax.set_extent(extent, crs=ccrs.PlateCarree())
        ax.add_feature(cfeature.OCEAN, facecolor="#d8eef7", zorder=0)
        ax.add_feature(cfeature.LAND, facecolor="#f1efe6", edgecolor="#9ca3af", linewidth=0.25, zorder=1)
        ax.add_feature(cfeature.COASTLINE, linewidth=0.9, edgecolor="#25313b", zorder=3)
        ax.add_feature(cfeature.BORDERS, linewidth=0.45, edgecolor="#6b7280", zorder=3)
        grille = ax.gridlines(draw_labels=True, linewidth=0.25, color="#6b7280", alpha=0.45)
        grille.top_labels = False
        grille.right_labels = False
    else:
        ax.set_xlim(extent[0], extent[1])
        ax.set_ylim(extent[2], extent[3])
        ax.set_facecolor("#d8eef7")
        ax.add_patch(Rectangle((9.0, 0.8), 3.7, 4.4, facecolor="#f1efe6", edgecolor="#9ca3af", linewidth=0.8))
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")
    ax.set_title(titre, fontsize=11, weight="bold")

projection = ccrs.PlateCarree() if CARTOPY_DISPONIBLE else None
fig = plt.figure(figsize=(12, 5.8), constrained_layout=True)
ax_context = fig.add_subplot(1, 2, 1, projection=projection) if CARTOPY_DISPONIBLE else fig.add_subplot(1, 2, 1)
ax_local = fig.add_subplot(1, 2, 2, projection=projection) if CARTOPY_DISPONIBLE else fig.add_subplot(1, 2, 2)
transform = ccrs.PlateCarree() if CARTOPY_DISPONIBLE else None
kwargs = {"transform": transform} if CARTOPY_DISPONIBLE else {}

# Carte régionale: vraie forme du Golfe de Guinée, pays voisins et emprise de la zone PAK/Kribi.
ajouter_fond_geographique(ax_context, extent_regional, "Contexte régional - Golfe de Guinée")
ax_context.add_patch(Rectangle((extent_local[0], extent_local[2]), extent_local[1] - extent_local[0], extent_local[3] - extent_local[2],
                               fill=False, edgecolor=GGOSSS_COLORS["shoreline"], linewidth=2.0, transform=transform))
for nom, x, y in [
    ("Nigéria", 8.0, 4.35),
    ("Cameroun", 10.1, 4.45),
    ("Guinée équatoriale", 10.8, 1.25),
    ("Gabon", 11.5, 1.0),
    ("Océan Atlantique", 7.7, 1.45),
]:
    ax_context.text(x, y, nom, fontsize=9, color=GGOSSS_COLORS["navy"] if nom == "Cameroun" else "#4b5563", weight="bold" if nom == "Cameroun" else "normal", **kwargs)
ax_context.text(9.61, 3.38, "PAK/Kribi", fontsize=9, color=GGOSSS_COLORS["shoreline"], weight="bold", **kwargs)

# Carte locale: localités côtières, secteurs pédagogiques P1-P4 et indice de pression cinq facteurs.
ajouter_fond_geographique(ax_local, extent_local, "Zoom PAK/Kribi - localités et secteurs")
if not localities.empty:
    localities_sorted = localities.sort_values("Lat")
    ax_local.plot(localities_sorted["Lon"], localities_sorted["Lat"], color="#1f2937", linewidth=1.0, alpha=0.65, zorder=4, **kwargs)
    ax_local.scatter(localities["Lon"], localities["Lat"], marker="^", color=GGOSSS_COLORS["navy"], s=46, zorder=5, **kwargs)
    for _, row in localities[localities["Name"].isin(["Lokoundje", "Londji", "Kribi", "Grand Batanga I", "Mboro", "Campo", "Ebodje", "Lolabe"])].iterrows():
        ax_local.text(row["Lon"] + 0.006, row["Lat"] + 0.006, row["Name"], fontsize=7.2, color="#111827", **kwargs)

points = ax_local.scatter(
    factors["Longitude"],
    factors["Latitude"],
    c=factors["coastal_dynamics_pressure_index"],
    s=235,
    cmap="YlOrRd",
    vmin=0,
    vmax=1,
    edgecolors="none",
    linewidths=0,
    zorder=6,
    **kwargs,
)
for _, row in factors.iterrows():
    ax_local.text(row["Longitude"] + 0.012, row["Latitude"] + 0.012, row["Point"], fontsize=9, weight="bold", color="#111827", **kwargs)

cbar = fig.colorbar(points, ax=ax_local, shrink=0.82)
cbar.set_label("Indice de pression cinq facteurs")
fig.suptitle("GGOSSS 2026 - Localisation PAK/Kribi avec fond continental Natural Earth", fontsize=13, weight="bold")
plt.show()

In [ ]:
# Bloc shoreline: classer chaque transect DSAS en érosion, stabilité ou accrétion.
def classify_rate(rate, threshold=0.5):
    if rate <= -threshold:
        return "erosion"
    if rate >= threshold:
        return "accretion"
    return "stable"

rates["dynamics_class"] = rates["endpoint_rate_myr"].apply(classify_rate)
summary = rates["dynamics_class"].value_counts().rename_axis("class").reset_index(name="transects")
summary

In [ ]:
# Bloc de visualisation shoreline: vérifier si l'érosion domine l'ensemble des transects.
fig, ax = plt.subplots()
rates["endpoint_rate_myr"].hist(ax=ax, bins=35, color=GGOSSS_COLORS["neutral"], edgecolor="white")
ax.axvline(-0.5, color=GGOSSS_COLORS["shoreline"], linestyle="--", label="erosion threshold")
ax.axvline(0.5, color="#2e7d32", linestyle="--", label="accretion threshold")
ax.set_xlabel("Endpoint rate (m/year)")
ax.set_ylabel("Transect count")
ax.set_title("GGOSSS 2026 · PAK shoreline-change rates")
ax.legend()
plt.show()

In [ ]:
# Bloc hotspots: classer les transects les plus érosifs avant l'agrégation par secteur.
hotspots = rates.nsmallest(10, "endpoint_rate_myr")[[
    "transect_id", "endpoint_rate_myr", "net_shoreline_movement_m", "shoreline_change_envelope_m", "endpoint_rate_unc_myr"
]]
hotspots

In [ ]:
# Bloc table cinq facteurs: examiner Shoreline, SWV, Tide, Wave et Wind par secteur pédagogique.
five_factor_cols = [
    "Point",
    "shoreline_mean_epr_myr",
    "shoreline_erosion_pct",
    "swv_mean_current_speed_mps",
    "tide_range_m",
    "wave_swh_p95_m",
    "wind_speed_p95_ms",
    "coastal_dynamics_pressure_index",
    "monitoring_priority",
]
factors[five_factor_cols].sort_values("coastal_dynamics_pressure_index", ascending=False)

In [ ]:
# Bloc diagnostic cinq facteurs: visualiser les scores normalisés avec les couleurs GGOSSS.
score_cols = ["shoreline_erosion_score", "swv_score", "tide_score", "wave_score", "wind_score"]
plot_df = factors.set_index("Point")[score_cols]
ax = plot_df.plot(
    kind="bar",
    figsize=(10, 4),
    color=[GGOSSS_COLORS["shoreline"], GGOSSS_COLORS["swv"], GGOSSS_COLORS["tide"], GGOSSS_COLORS["wave"], GGOSSS_COLORS["wind"]],
)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Normalized pressure score")
ax.set_title("GGOSSS 2026 · Five-factor coastal dynamics diagnosis")
ax.legend(["Shoreline", "SWV", "Tide", "Wave", "Wind"], ncol=5, loc="upper center", bbox_to_anchor=(0.5, -0.18))
plt.tight_layout()
plt.show()

In [ ]:
# Bloc pondération: comparer trois scénarios défendables de priorisation du suivi.
weight_scenarios = {
    "balanced": {"shoreline_erosion_score": 0.20, "swv_score": 0.20, "tide_score": 0.20, "wave_score": 0.20, "wind_score": 0.20},
    "observed_erosion_dominant": {"shoreline_erosion_score": 0.50, "swv_score": 0.15, "tide_score": 0.10, "wave_score": 0.15, "wind_score": 0.10},
    "hydrodynamic_forcing": {"shoreline_erosion_score": 0.20, "swv_score": 0.25, "tide_score": 0.15, "wave_score": 0.25, "wind_score": 0.15},
}

scenario_results = factors[["Point"]].copy()
for scenario, weights in weight_scenarios.items():
    scenario_results[scenario] = sum(factors[col] * weight for col, weight in weights.items())

scenario_results = scenario_results.set_index("Point")
scenario_results

In [ ]:
# Bloc comparaison de scénarios: tester si le secteur prioritaire change lorsque les poids changent.
ax = scenario_results.plot(kind="bar", figsize=(9, 4), color=[GGOSSS_COLORS["navy"], GGOSSS_COLORS["shoreline"], GGOSSS_COLORS["wave"]])
ax.set_ylim(0, 1)
ax.set_ylabel("Weighted pressure score")
ax.set_title("GGOSSS 2026 · Monitoring priority under alternative weights")
plt.xticks(rotation=0)
plt.show()

print("Priority sector by scenario:")
print(scenario_results.idxmax())

In [ ]:
# Bloc incertitude: identifier les boîtes CoastSat où les estimations de pente de plage sont peu contraintes.
boxes[["box", "transects", "shorelines", "dateMin", "dateMax", "slopeMedian", "ciWidthMedian", "poorPct"]].sort_values("poorPct", ascending=False)

In [ ]:
# Bloc synthèse du suivi: transformer l'analyse en table courte de recommandation pour la discussion.
def assign_action(score):
    if score >= 0.60:
        return "priority field check + shoreline validation"
    if score >= 0.40:
        return "targeted remote-sensing review + selective field check"
    return "routine remote-sensing monitoring"

recommendation = factors[["Point", "coastal_dynamics_pressure_index", "monitoring_priority"]].copy()
recommendation["recommended_action"] = recommendation["coastal_dynamics_pressure_index"].apply(assign_action)
recommendation.sort_values("coastal_dynamics_pressure_index", ascending=False)

## Mini-challenge: recommandation de groupe en 5 minutes

Chaque groupe doit formuler une recommandation concise:

1. Choose one priority sector or transect group.
2. Justify it using at least **three** of the five factors: Shoreline, SWV, Tide, Wave, Wind.
3. State one uncertainty that could change the recommendation.
4. Propose one field observation or additional dataset to collect next.

L'indice de pression est un outil pédagogique, pas un indice de risque validé. Une bonne réponse critique la pondération et explique l'incertitude.